# LLM Single vs Batch Prompt Semantic Judgment Compare

This notebook compares whether LLM semantic disambiguation changes when the same records are judged one-by-one versus grouped into a batched prompt. It does not run FFT.

The important control is that single and batch runs use separate cache files, so the batch run cannot simply reuse single-prompt answers.

In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import pickle
import sys
import re
import time

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "local_llm.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from local_llm import LocalLLMClient, LocalLLMConfig
from text_processing import normalize_text
from llm_semantic_labeler import (
    DEFAULT_CONTEXT_WORD_WINDOW,
    DEFAULT_MAX_TOKENS,
    choose_wikidata_candidate_with_llm,
    choose_wikidata_candidates_with_llm_batch,
)
from utils import (
    build_wikidata_candidate_bank,
    definition_to_hypothesis,
    load_wikidata_definition_candidates,
)

In [2]:
# Data and sampling settings. This follows hotpotqa_fft_span_compare.ipynb:
# load the full HotpotQA span scan store, then lookup occurrences for TARGET_SPAN.
HOTPOT_SCAN_STORE_PATH = REPO_ROOT / "data/hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl"
TARGET_SPAN = "director"
QUERY_KIND = None  # set to "phrase" or "token" if you want one kind only
INPUT_SAMPLE_LIMIT = 150
PROMPT_CONTEXT_MODE = "sentence_neighbors"  # "sentence", "sentence_neighbors", "full_text"
MARK_TARGET = False

# Candidate-meaning settings. These mirror the style used in hotpotqa_fft_span_compare.ipynb.
WIKIDATA_CANDIDATE_LIMIT = 8
USE_DETAILED_DESCRIPTION = False
REQUIRE_DETAILED_DESCRIPTION = True
EXACT_MATCH_TEXT = True
FILTER_NAME = True

# Candidate-merge settings. When enabled, an LLM pass merges noisy/duplicated
# Wikidata candidate senses into a smaller coarse retrieval set (the same merge
# prompt used in wikidata_llm_candidate_merge_experiment.ipynb) BEFORE the
# single/batch disambiguation runs. The single/batch comparison then selects from
# the merged senses instead of the raw Wikidata candidates.
MERGE_CANDIDATES = True
MERGE_MAX_TOKENS = 2048

# LLM settings
LLM_PROVIDER = "openai"  # "openai" or "local"
LLM_MODEL = "gpt-5.4-mini"
LLM_API_KEY_FILE = "API_KEY"
LLM_CONTEXT_WORD_WINDOW = DEFAULT_CONTEXT_WORD_WINDOW
LLM_MAX_TOKENS = 2048
LLM_BATCH_SIZE = 10
LLM_BATCH_MIN_SIZE = 1

# Keep this False until you inspect the sampled records/candidates.
RUN_LLM = True

# Use separate caches so the batch-prompt run cannot reuse single-prompt judgments.
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
SINGLE_CACHE_PATH = REPO_ROOT / f"cache/llm_prompt_compare_single_{RUN_TAG}.sqlite3"
BATCH_CACHE_PATH = REPO_ROOT / f"cache/llm_prompt_compare_batch_{RUN_TAG}.sqlite3"

In [3]:
def load_hotpot_scan_store(scan_store_path: Path = HOTPOT_SCAN_STORE_PATH):
    if not scan_store_path.exists():
        raise FileNotFoundError(
            f"HotpotQA scan store not found at {scan_store_path}. Please prepare the scan cache first."
        )
    with scan_store_path.open("rb") as handle:
        store = pickle.load(handle)
    print(f"Loaded scan-only store from {scan_store_path}")
    print(store.get("stats"))
    return store


def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def _find_left_boundary(text: str, index: int) -> int:
    return max(
        text.rfind(".", 0, index),
        text.rfind("!", 0, index),
        text.rfind("?", 0, index),
    )


def _find_right_boundary(text: str, index: int) -> int:
    right_candidates = [
        text.find(".", index),
        text.find("!", index),
        text.find("?", index),
    ]
    right_candidates = [idx for idx in right_candidates if idx != -1]
    return len(text) if not right_candidates else min(right_candidates) + 1


def _build_context_from_bounds(cleaned_text, span, context_start, context_end):
    start_char, end_char = span
    context_raw = cleaned_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(cleaned_text), end_char + 120)
        context_raw = cleaned_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def extract_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)
    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_neighbor_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)

    if context_start > 0:
        previous_boundary = _find_left_boundary(cleaned_text, max(0, context_start - 1))
        context_start = 0 if previous_boundary == -1 else previous_boundary + 1

    if context_end < len(cleaned_text):
        context_end = _find_right_boundary(cleaned_text, context_end)

    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_full_context(cleaned_text, span):
    start_char, end_char = span
    context_raw = cleaned_text
    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - left_trim
    local_end = end_char - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def extract_prompt_context(cleaned_text, span, prompt_context_mode="sentence"):
    if prompt_context_mode == "sentence":
        return extract_sentence_context(cleaned_text, span)
    if prompt_context_mode == "full_text":
        return extract_full_context(cleaned_text, span)
    if prompt_context_mode == "sentence_neighbors":
        return extract_neighbor_sentence_context(cleaned_text, span)
    raise ValueError(
        f"Unsupported prompt_context_mode={prompt_context_mode!r}. Use 'sentence', 'sentence_neighbors', or 'full_text'."
    )


def build_hotpot_prompt(
    record,
    query_text,
    mark_target=False,
    left_marker="[TGT]",
    right_marker="[/TGT]",
    prompt_context_mode="sentence",
):
    context_info = extract_prompt_context(
        record["cleaned_text"],
        record["span"],
        prompt_context_mode=prompt_context_mode,
    )
    context_text = context_info["context_text"]
    local_start, local_end = context_info["local_span"]
    prompt_context = context_text

    if mark_target:
        prompt_context = (
            f"{context_text[:local_start]}{left_marker} {context_text[local_start:local_end]} {right_marker}{context_text[local_end:]}"
        )

    prompt_text = (
        f"Context: {prompt_context}\n"
        f"Target word: {query_text.strip()}\n\n"
        f'Question: What does "{query_text.strip()}" mean in this context?'
    )

    return {
        "context_text": context_text,
        "matched_text": context_text[local_start:local_end],
        "local_span": (local_start, local_end),
        "prompt_text": prompt_text,
    }


def collect_hotpot_prompt_records(
    store,
    query_text,
    *,
    kind=None,
    max_records=None,
    prompt_context_mode="sentence_neighbors",
    mark_target=False,
):
    query_records = lookup_records(
        store,
        query_text,
        kind=kind,
        include_text=True,
        include_cleaned_text=True,
    )
    total_available = len(query_records)
    if max_records is not None:
        query_records = query_records[: int(max_records)]

    rows = []
    for record in query_records:
        prompt_info = build_hotpot_prompt(
            record,
            query_text,
            mark_target=mark_target,
            prompt_context_mode=prompt_context_mode,
        )
        rows.append({
            "record_id": len(rows),
            "document_idx": int(record["document_idx"]),
            "title": store["documents"][record["document_idx"]].get("title"),
            "kind": record.get("kind"),
            "span": tuple(record["span"]),
            "span_text": query_text,
            "matched_text": prompt_info["matched_text"],
            "context_text": prompt_info["context_text"],
            "local_span": prompt_info["local_span"],
            "prompt_text": prompt_info["prompt_text"],
        })
    return rows, total_available


embedding_store = load_hotpot_scan_store()
records, total_available_records = collect_hotpot_prompt_records(
    embedding_store,
    TARGET_SPAN,
    kind=QUERY_KIND,
    max_records=INPUT_SAMPLE_LIMIT,
    prompt_context_mode=PROMPT_CONTEXT_MODE,
    mark_target=MARK_TARGET,
)

print(
    f"Collected {len(records)} records for target span {TARGET_SPAN!r} "
    f"from {total_available_records} available scan-store occurrences."
)
display(pd.DataFrame(records).head(10))

Loaded scan-only store from /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl
{'num_documents': 66581, 'num_unique_terms': 634624, 'num_phrase_occurrences': 1007780, 'num_token_occurrences': 584999, 'num_total_occurrences': 1592779}
Collected 150 records for target span 'director' from 1148 available scan-store occurrences.


,record_id,document_idx,title,kind,span,span_text,matched_text,context_text,local_span,prompt_text
0,0,4,Ed Wood,token,"(117, 125)",director,director,"Edward Davis Wood Jr. (october 10, 1924 – dece...","(117, 125)","Context: Edward Davis Wood Jr. (october 10, 19..."
1,1,113,David Weissman,token,"(37, 45)",director,director,David Weissman is a screenwriter and director....,"(37, 45)",Context: David Weissman is a screenwriter and ...
2,2,174,Ambroise Thomas,token,"(172, 180)",director,director,Charles Louis Ambroise Thomas (5 august 1811 –...,"(172, 180)",Context: Charles Louis Ambroise Thomas (5 augu...
3,3,176,Ferdinando Provesi,token,"(401, 409)",director,director,bartolomeo cathedral in busseto (the town very...,"(120, 128)",Context: bartolomeo cathedral in busseto (the ...
4,4,179,United States Assistant Secretary of State,token,"(576, 584)",director,director,Assistant Secretaries usually manage individua...,"(185, 193)",Context: Assistant Secretaries usually manage ...
5,5,182,Steven K. Galson,token,"(693, 701)",director,director,he is a retired rear admiral in the United Sta...,"(360, 368)",Context: he is a retired rear admiral in the U...
6,6,183,Robert R. Hood,token,"(561, 569)",director,director,past roles with the federal government include...,"(363, 371)",Context: past roles with the federal governmen...
7,7,323,Heidi Ewing,token,"(17, 25)",director,director,"Heidi Ewing is a director, producer, and write...","(17, 25)","Context: Heidi Ewing is a director, producer, ..."
8,8,323,Heidi Ewing,token,"(942, 950)",director,director,Norman Lear: Just Another Version of you was t...,"(184, 192)",Context: Norman Lear: Just Another Version of ...
9,9,359,Antonino Lo Surdo,token,"(240, 248)",director,director,Antonino Lo Surdo (4 february 1880 in syracuse...,"(240, 248)",Context: Antonino Lo Surdo (4 february 1880 in...


In [4]:
FALLBACK_CANDIDATES = {
    "space": [
        {"entity_id": "Q107", "label": "space", "description": "boundless three-dimensional extent", "definition": "boundless three-dimensional extent"},
        {"entity_id": "Q4169", "label": "outer space", "description": "region beyond Earth atmosphere", "definition": "the expanse that exists beyond Earth and its atmosphere"},
        {"entity_id": "Q13226383", "label": "space", "description": "blank or empty area", "definition": "an empty area or interval available for use"},
    ],
    "film": [
        {"entity_id": "Q11424", "label": "film", "description": "sequence of images that give the impression of movement", "definition": "a motion picture or movie"},
        {"entity_id": "Q20937557", "label": "photographic film", "description": "strip or sheet of transparent plastic film base", "definition": "a light-sensitive material used in photography"},
    ],
    "song": [
        {"entity_id": "Q7366", "label": "song", "description": "musical composition for voice", "definition": "a musical composition intended to be sung"},
        {"entity_id": "Q2188189", "label": "bird vocalization", "description": "bird song or call", "definition": "sounds produced by birds for communication"},
    ],
    "bank": [
        {"entity_id": "Q22687", "label": "bank", "description": "financial institution", "definition": "a financial institution that accepts deposits and makes loans"},
        {"entity_id": "Q3089714", "label": "river bank", "description": "terrain alongside the bed of a river", "definition": "the land alongside a river or other body of water"},
    ],
}


def fallback_candidate_bank(target_span: str):
    candidates = FALLBACK_CANDIDATES.get(target_span.lower())
    if not candidates:
        raise ValueError(f"No fallback candidates configured for {target_span!r}.")
    return [
        {
            **candidate,
            "definition_source": "fallback",
            "hypothesis": definition_to_hypothesis(candidate["definition"]),
        }
        for candidate in candidates
    ]


def load_candidate_bank(target_span: str):
    try:
        candidates_df, definition_column = load_wikidata_definition_candidates(
            target_span,
            use_detailed_description=USE_DETAILED_DESCRIPTION,
            exact_match_text=EXACT_MATCH_TEXT,
            filter_name=FILTER_NAME,
            require_detailed_description=REQUIRE_DETAILED_DESCRIPTION,
            limit=WIKIDATA_CANDIDATE_LIMIT,
            target_candidate_count=WIKIDATA_CANDIDATE_LIMIT,
        )
        bank = build_wikidata_candidate_bank(candidates_df, definition_column)
        source = f"wikidata:{definition_column}"
    except Exception as exc:
        print(f"Wikidata candidate loading failed, using fallback candidates: {exc}")
        bank = fallback_candidate_bank(target_span)
        source = "fallback"
    if len(bank) < 2:
        print("Warning: fewer than two candidate meanings; prompt-mode differences may be uninformative.")
    return bank, source


candidate_bank, candidate_source = load_candidate_bank(TARGET_SPAN)
print(f"Candidate source: {candidate_source}; count={len(candidate_bank)}")
display(pd.DataFrame(candidate_bank)[["entity_id", "label", "description", "definition_source", "definition"]])

Candidate source: wikidata:description; count=4


,entity_id,label,description,definition_source,definition
0,Q3455803,director,director of a creative work,description,director of a creative work
1,Q1162163,director,person who leads a particular area of a compan...,description,person who leads a particular area of a compan...
2,Q2526255,film director,person who controls the artistic and dramatic ...,description,person who controls the artistic and dramatic ...
3,Q3387717,theatre director,person overseeing the mounting of a theatre pr...,description,person overseeing the mounting of a theatre pr...


In [5]:
llm_client = None
if RUN_LLM:
    api_key_file = Path(LLM_API_KEY_FILE)
    if not api_key_file.is_absolute():
        api_key_file = REPO_ROOT / api_key_file
    llm_config = LocalLLMConfig.from_env(
        provider=LLM_PROVIDER,
        model=LLM_MODEL,
        api_key_file=str(api_key_file),
    )
    llm_client = LocalLLMClient(llm_config)
    provider = llm_client.config.provider
    model = llm_client.config.model
else:
    provider = LLM_PROVIDER
    model = LLM_MODEL

print({
    "run_llm": RUN_LLM,
    "provider": provider,
    "model": model,
    "single_cache": str(SINGLE_CACHE_PATH),
    "batch_cache": str(BATCH_CACHE_PATH),
})

{'run_llm': True, 'provider': 'openai', 'model': 'gpt-5.4-mini', 'single_cache': '/home/xiaoyue/LiteSemRAG/cache/llm_prompt_compare_single_20260528_205757.sqlite3', 'batch_cache': '/home/xiaoyue/LiteSemRAG/cache/llm_prompt_compare_batch_20260528_205757.sqlite3'}


## Optional: LLM Merge Of Candidate Senses

When `MERGE_CANDIDATES` is True, run a single LLM pass that merges semantically close or duplicated Wikidata candidate senses into a smaller coarse retrieval set, using the same conservative merge prompt as `wikidata_llm_candidate_merge_experiment.ipynb`. This reuses the notebook's `llm_client`.

The raw Wikidata bank is kept in `raw_candidate_bank`; `candidate_bank` is overwritten with the merged senses so the single/batch disambiguation cells below select from the merged inventory. This merge call runs before the single/batch token counters are read, so its token cost is excluded from `single_tokens_delta` / `batch_tokens_delta`.

In [6]:
import json

# Conservative merge prompt, copied from wikidata_llm_candidate_merge_experiment.ipynb.
MERGE_SYSTEM_PROMPT = """You are a conservative lexical-sense merger for a semantic retrieval system.
Your job is to reduce noisy Wikidata candidate senses to a small set of coarse retrieval meanings.
Prefer merging over splitting when candidates describe the same broad concept, role, entity type, or function.
Do not preserve fine-grained domain, institution, jurisdiction, title, or wording differences unless they change what evidence should be retrieved.
When uncertain, merge the candidates and write a broader description.
Return only valid JSON."""


MERGE_EXAMPLES = [
    {
        "word": "president",
        "candidate_senses": [
            {"candidate_id": 1, "label": "president", "hypothesis": "It refers to leader of a country or part of a country."},
            {"candidate_id": 2, "label": "president", "hypothesis": "It refers to leader of an organization."},
            {"candidate_id": 3, "label": "speaker", "hypothesis": "It refers to presiding officer of a legislative body."},
            {"candidate_id": 4, "label": "chancellor", "hypothesis": "It refers to leader of a university or college."},
        ],
        "expected_merge": {
            "canonical_label": "leader or presiding officer",
            "merged_description": "A person who leads or presides over a country, organization, legislative body, university, or similar institution.",
            "source_candidate_ids": [1, 2, 3, 4],
        },
        "rationale": "These are institutional leadership or presiding roles. The differences are title/domain variants, not separate retrieval meanings for a generic word-sense index.",
    },
    {
        "word": "bank",
        "candidate_senses": [
            {"candidate_id": 1, "label": "bank", "hypothesis": "It refers to a financial institution."},
            {"candidate_id": 2, "label": "river bank", "hypothesis": "It refers to land alongside a river."},
        ],
        "expected_split": [1, 2],
        "rationale": "These meanings retrieve different kinds of evidence and should stay separate.",
    },
    {
        "word": "apple",
        "candidate_senses": [
            {"candidate_id": 1, "label": "apple", "hypothesis": "It refers to an edible fruit."},
            {"candidate_id": 2, "label": "Apple Inc.", "hypothesis": "It refers to a technology company."},
        ],
        "expected_split": [1, 2],
        "rationale": "A fruit and a company are different entity types and should stay separate.",
    },
]


def build_merge_prompt(word, candidates):
    payload = {
        "word": word,
        "candidate_senses": [
            {
                "candidate_id": candidate["candidate_id"],
                "label": candidate["label"],
                "hypothesis": candidate["hypothesis"],
            }
            for candidate in candidates
        ],
    }
    return f"""Merge candidate senses for the target word into coarse retrieval-oriented meanings.

Goal:
Create the smallest useful sense inventory for retrieval. These merged descriptions will later be used by a cross-encoder, so avoid distinctions that are too subtle for short context snippets.

Default bias:
- Merge by broad semantic function, not by Wikidata entity granularity.
- Merge title/domain variants when they are instances of the same role or concept.
- Merge specific subtypes into their broader parent sense unless the subtype changes the entity type or expected evidence.
- If two candidates could both match the same ordinary sentence about the target word, merge them.
- When uncertain, merge.

Split only when:
1. The meanings are genuinely different entity types or concepts, such as fruit vs company or financial bank vs river bank.
2. Keeping them together would make clearly wrong documents look relevant.
3. The distinction is likely obvious from short local context, not just from specialist wording.

Discard only when:
1. The candidate is not a plausible sense of the target word.
2. The candidate is too vague to add value and cannot be merged into a broader valid sense.

Examples:
{json.dumps(MERGE_EXAMPLES, ensure_ascii=False, indent=2)}

Output only valid JSON with keys: word, merged_senses, discarded_candidates, notes.

Each merged_senses item must contain:
- sense_id: a short stable id such as s1, s2, s3
- canonical_label: short label for the merged sense
- merged_description: one sentence describing the broad merged meaning
- source_candidate_ids: list of integer candidate_id values that were merged
- merge_rationale: one short sentence explaining why these candidates belong together or why the sense stayed separate

Each discarded_candidates item must contain:
- candidate_id
- reason

Candidate data:
{json.dumps(payload, ensure_ascii=False, indent=2)}"""


def _merged_entity_id(source_candidate_ids, raw_bank, sense_id):
    entity_ids = []
    for candidate_id in source_candidate_ids or []:
        try:
            index = int(candidate_id) - 1
        except (TypeError, ValueError):
            continue
        if 0 <= index < len(raw_bank):
            entity_id = str(raw_bank[index].get("entity_id") or "").strip()
            if entity_id:
                entity_ids.append(entity_id)
    if entity_ids:
        return "+".join(dict.fromkeys(entity_ids))
    return f"merged:{sense_id}"


def merge_candidate_bank_with_llm(word, raw_bank, *, client, max_tokens):
    # candidate_id is the 1-based position into raw_bank, so the LLM output maps back cleanly.
    prompt_candidates = [
        {
            "candidate_id": index,
            "label": candidate.get("label"),
            "hypothesis": candidate.get("hypothesis"),
        }
        for index, candidate in enumerate(raw_bank, start=1)
    ]
    raw_response = client.complete(
        build_merge_prompt(word, prompt_candidates),
        system_prompt=MERGE_SYSTEM_PROMPT,
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=max_tokens,
    )
    merge_result = json.loads(raw_response)

    merged_bank = []
    for sense in merge_result.get("merged_senses", []) or []:
        description = " ".join(str(sense.get("merged_description") or "").split()).strip()
        if not description:
            continue
        sense_id = sense.get("sense_id") or f"s{len(merged_bank) + 1}"
        source_candidate_ids = sense.get("source_candidate_ids") or []
        label = str(sense.get("canonical_label") or word).strip()
        merged_bank.append(
            {
                "entity_id": _merged_entity_id(source_candidate_ids, raw_bank, sense_id),
                "label": label,
                "description": description,
                "definition": description,
                "definition_source": "llm_merged",
                "hypothesis": definition_to_hypothesis(description),
                "sense_id": sense_id,
                "source_candidate_ids": list(source_candidate_ids),
                "merge_rationale": sense.get("merge_rationale"),
            }
        )
    return merged_bank, merge_result


raw_candidate_bank = candidate_bank
merge_result = None
if MERGE_CANDIDATES and RUN_LLM:
    merged_bank, merge_result = merge_candidate_bank_with_llm(
        TARGET_SPAN,
        raw_candidate_bank,
        client=llm_client,
        max_tokens=MERGE_MAX_TOKENS,
    )
    if len(merged_bank) >= 1:
        candidate_bank = merged_bank
        print(f"Merged {len(raw_candidate_bank)} -> {len(candidate_bank)} candidate senses.")
        display(pd.DataFrame(candidate_bank)[
            ["entity_id", "label", "description", "source_candidate_ids", "merge_rationale"]
        ])
        discarded = merge_result.get("discarded_candidates") or []
        if discarded:
            print("Discarded candidates:")
            display(pd.DataFrame(discarded))
        notes = merge_result.get("notes")
        if notes:
            print(f"Merge notes: {notes}")
    else:
        print("LLM merge returned no usable senses; keeping the raw candidate bank.")
elif MERGE_CANDIDATES and not RUN_LLM:
    print("MERGE_CANDIDATES is True but RUN_LLM is False; keeping raw candidate bank.")
else:
    print("Candidate merge disabled (MERGE_CANDIDATES=False); using raw candidate bank.")

Merged 4 -> 2 candidate senses.


,entity_id,label,description,source_candidate_ids,merge_rationale
0,Q3455803+Q2526255+Q3387717,creative director,A person who directs the artistic or creative ...,"[1, 3, 4]",These are all creative-production directing ro...
1,Q1162163,organizational director,"A person who leads a department, division, or ...",[2],This is a distinct organizational leadership r...


Merge notes: Merged the film and theatre variants into a broad creative-direction sense because the distinction is usually too fine for retrieval.


In [7]:
def run_single_prompt_judgments(records, candidate_bank, *, client, cache_path):
    results = []
    started = time.perf_counter()
    for idx, record in enumerate(records, start=1):
        result = choose_wikidata_candidate_with_llm(
            span_text=record["span_text"],
            context_text=record["context_text"],
            matched_text=record.get("matched_text"),
            local_span=record.get("local_span"),
            candidate_bank=candidate_bank,
            cache_path=cache_path,
            context_word_window=LLM_CONTEXT_WORD_WINDOW,
            client=client,
            max_tokens=LLM_MAX_TOKENS,
        )
        selected = result["selected_candidate"]
        results.append({
            "record_id": record["record_id"],
            "selected_index": result["selected_index"],
            "selected_entity_id": selected.get("entity_id"),
            "selected_label": selected.get("label"),
            "selected_description": selected.get("description"),
            "reason": result.get("reason"),
            "cache_hit": bool(result.get("cache_hit")),
        })
        if idx % 10 == 0 or idx == len(records):
            print(f"single prompt: {idx}/{len(records)}")
    return {
        "results": results,
        "wall_time_sec": time.perf_counter() - started,
        "total_tokens_after": client.total_tokens,
    }


def _run_batch_prompt_slice(records_slice, candidate_bank, *, client, cache_path, batch_size):
    try:
        return choose_wikidata_candidates_with_llm_batch(
            records=records_slice,
            candidate_bank=candidate_bank,
            cache_path=cache_path,
            context_word_window=LLM_CONTEXT_WORD_WINDOW,
            client=client,
            max_tokens=LLM_MAX_TOKENS,
            max_batch_size=batch_size,
        )
    except Exception as exc:
        if batch_size <= LLM_BATCH_MIN_SIZE or len(records_slice) <= 1:
            raise RuntimeError(
                f"Batch prompt failed at batch_size={batch_size}, "
                f"record_ids={[r['record_id'] for r in records_slice]}"
            ) from exc
        smaller = max(LLM_BATCH_MIN_SIZE, batch_size // 2)
        print(
            f"Batch prompt failed for {len(records_slice)} records at batch_size={batch_size}; "
            f"retrying with batch_size={smaller}. Error: {type(exc).__name__}: {exc}"
        )
        outputs = []
        for offset in range(0, len(records_slice), smaller):
            outputs.extend(
                _run_batch_prompt_slice(
                    records_slice[offset:offset + smaller],
                    candidate_bank,
                    client=client,
                    cache_path=cache_path,
                    batch_size=smaller,
                )
            )
        return outputs


def run_batch_prompt_judgments(records, candidate_bank, *, client, cache_path, batch_size):
    started = time.perf_counter()
    raw_results = []
    processed = 0
    for batch_start in range(0, len(records), batch_size):
        batch_records = records[batch_start:batch_start + batch_size]
        raw_results.extend(
            _run_batch_prompt_slice(
                batch_records,
                candidate_bank,
                client=client,
                cache_path=cache_path,
                batch_size=batch_size,
            )
        )
        processed += len(batch_records)
        print(f"batch prompt: {processed}/{len(records)}")

    results = []
    for record, result in zip(records, raw_results):
        selected = result["selected_candidate"]
        results.append({
            "record_id": record["record_id"],
            "selected_index": result["selected_index"],
            "selected_entity_id": selected.get("entity_id"),
            "selected_label": selected.get("label"),
            "selected_description": selected.get("description"),
            "reason": result.get("reason"),
            "cache_hit": bool(result.get("cache_hit")),
        })
    return {
        "results": results,
        "wall_time_sec": time.perf_counter() - started,
        "total_tokens_after": client.total_tokens,
    }

In [8]:
if RUN_LLM:
    tokens_before_single = llm_client.total_tokens
    single_result = run_single_prompt_judgments(
        records,
        candidate_bank,
        client=llm_client,
        cache_path=SINGLE_CACHE_PATH,
    )
    tokens_after_single = llm_client.total_tokens

    batch_result = run_batch_prompt_judgments(
        records,
        candidate_bank,
        client=llm_client,
        cache_path=BATCH_CACHE_PATH,
        batch_size=LLM_BATCH_SIZE,
    )
    tokens_after_batch = llm_client.total_tokens

    print({
        "single_wall_time_sec": round(single_result["wall_time_sec"], 2),
        "batch_wall_time_sec": round(batch_result["wall_time_sec"], 2),
        "single_tokens_delta": tokens_after_single - tokens_before_single,
        "batch_tokens_delta": tokens_after_batch - tokens_after_single,
    })
else:
    print("RUN_LLM is False. Inspect records/candidates, then set RUN_LLM=True and rerun this cell.")

single prompt: 10/150
single prompt: 20/150
single prompt: 30/150
single prompt: 40/150
single prompt: 50/150
single prompt: 60/150
single prompt: 70/150
single prompt: 80/150
single prompt: 90/150
single prompt: 100/150
single prompt: 110/150
single prompt: 120/150
single prompt: 130/150
single prompt: 140/150
single prompt: 150/150
batch prompt: 10/150
batch prompt: 20/150
batch prompt: 30/150
batch prompt: 40/150
batch prompt: 50/150
batch prompt: 60/150
batch prompt: 70/150
batch prompt: 80/150
batch prompt: 90/150
batch prompt: 100/150
batch prompt: 110/150
batch prompt: 120/150
batch prompt: 130/150
batch prompt: 140/150
batch prompt: 150/150
{'single_wall_time_sec': 123.61, 'batch_wall_time_sec': 33.94, 'single_tokens_delta': 53692, 'batch_tokens_delta': 20875}


In [9]:
def build_comparison_frame(records, single_result, batch_result):
    record_df = pd.DataFrame(records)
    single_df = pd.DataFrame(single_result["results"]).add_prefix("single_").rename(columns={"single_record_id": "record_id"})
    batch_df = pd.DataFrame(batch_result["results"]).add_prefix("batch_").rename(columns={"batch_record_id": "record_id"})
    merged = record_df.merge(single_df, on="record_id").merge(batch_df, on="record_id")
    merged["same_selected_index"] = merged["single_selected_index"] == merged["batch_selected_index"]
    merged["same_entity_id"] = merged["single_selected_entity_id"] == merged["batch_selected_entity_id"]
    merged["same_description"] = merged["single_selected_description"] == merged["batch_selected_description"]
    return merged


if RUN_LLM:
    comparison_df = build_comparison_frame(records, single_result, batch_result)
    summary = {
        "record_count": len(comparison_df),
        "same_selected_index": int(comparison_df["same_selected_index"].sum()),
        "same_entity_id": int(comparison_df["same_entity_id"].sum()),
        "same_description": int(comparison_df["same_description"].sum()),
        "agreement_rate_by_index": float(comparison_df["same_selected_index"].mean()),
        "agreement_rate_by_entity_id": float(comparison_df["same_entity_id"].mean()),
    }
    print(summary)
    display(comparison_df[[
        "record_id", "title", "matched_text", "context_text",
        "single_selected_index", "single_selected_description",
        "batch_selected_index", "batch_selected_description",
        "same_selected_index",
    ]])
else:
    print("Run the LLM cell first.")

{'record_count': 150, 'same_selected_index': 148, 'same_entity_id': 148, 'same_description': 148, 'agreement_rate_by_index': 0.9866666666666667, 'agreement_rate_by_entity_id': 0.9866666666666667}


,record_id,title,matched_text,context_text,single_selected_index,single_selected_description,batch_selected_index,batch_selected_description,same_selected_index
0,0,Ed Wood,director,"Edward Davis Wood Jr. (october 10, 1924 – dece...",0,A person who directs the artistic or creative ...,0,A person who directs the artistic or creative ...,True
1,1,David Weissman,director,David Weissman is a screenwriter and director....,0,A person who directs the artistic or creative ...,0,A person who directs the artistic or creative ...,True
2,2,Ambroise Thomas,director,Charles Louis Ambroise Thomas (5 august 1811 –...,1,"A person who leads a department, division, or ...",1,"A person who leads a department, division, or ...",True
3,3,Ferdinando Provesi,director,bartolomeo cathedral in busseto (the town very...,1,"A person who leads a department, division, or ...",1,"A person who leads a department, division, or ...",True
4,4,United States Assistant Secretary of State,director,Assistant Secretaries usually manage individua...,1,"A person who leads a department, division, or ...",1,"A person who leads a department, division, or ...",True
...,...,...,...,...,...,...,...,...,...
145,145,Vito Frazzi,director,"he was born in San Secondo Parmense, and studi...",1,"A person who leads a department, division, or ...",1,"A person who leads a department, division, or ...",True
146,146,Robert Cohen (writer),director,robert rob cohen is a canadian comedy writer a...,0,A person who directs the artistic or creative ...,0,A person who directs the artistic or creative ...,True
147,147,Martin Kunert,director,Martin Kunert (marcin stanisław kunert-dziewan...,0,A person who directs the artistic or creative ...,0,A person who directs the artistic or creative ...,True
148,148,Helen Hunt,director,"Helen Elizabeth Hunt (born june 15, 1963) is a...",0,A person who directs the artistic or creative ...,0,A person who directs the artistic or creative ...,True


In [10]:
if RUN_LLM:
    disagreement_df = comparison_df[~comparison_df["same_selected_index"]].copy()
    print(f"Disagreements: {len(disagreement_df)} / {len(comparison_df)}")
    display(disagreement_df[[
        "record_id", "title", "matched_text", "context_text",
        "single_selected_index", "single_selected_description", "single_reason",
        "batch_selected_index", "batch_selected_description", "batch_reason",
    ]])
else:
    print("Run the LLM cell first.")

Disagreements: 2 / 150


,record_id,title,matched_text,context_text,single_selected_index,single_selected_description,single_reason,batch_selected_index,batch_selected_description,batch_reason
89,89,Phillip Boykin,director,he was awarded the Theater World Award for his...,0,A person who directs the artistic or creative ...,The context says he is the founder and directo...,1,"A person who leads a department, division, or ...",Founder and head of a gospel group.
96,96,Eve Unsell,director,some of her most famous screen writes turned i...,0,A person who directs the artistic or creative ...,The context lists literary and theatrical role...,1,"A person who leads a department, division, or ...",Company director refers to organizational lead...


In [11]:
if RUN_LLM:
    output_dir = REPO_ROOT / "cache/llm_prompt_compare"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"single_vs_batch_{TARGET_SPAN}_{RUN_TAG}.csv"
    comparison_df.to_csv(output_path, index=False)
    print(f"Saved comparison CSV to {output_path}")
else:
    print("Run the LLM cell first.")

Saved comparison CSV to /home/xiaoyue/LiteSemRAG/cache/llm_prompt_compare/single_vs_batch_director_20260528_205757.csv
